## Setup & Authentication

 Install required libraries (kaggle, pandas)    

In [67]:
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi
import os
import sys
import sqlite3

Create connection for sqlite

In [68]:
with sqlite3.connect("supermarket_sales.db", timeout=30) as conn:
    conn.execute("PRAGMA journal_mode=WAL;")

 Authenticate with Kaggle API via kaggle.json  

In [69]:
api = KaggleApi()
api.authenticate()

In [70]:
folder_path = r'\Supermarket_Project\Data\raw'

bronze_path = os.path.join(folder_path, 'bronze')
silver_path = os.path.join(folder_path, 'silver')
gold_path   = os.path.join(folder_path, 'gold')
quarantine_path = os.path.join(folder_path, 'quarantine')


for path in [bronze_path, silver_path, gold_path, quarantine_path]:
    os.makedirs(path, exist_ok=True)

## Raw  / Bronze Layer

 Download dataset via Kaggle API


In [71]:
dataset_files = api.dataset_list_files('lovishbansal123/sales-of-a-supermarket').files

## Loop through Kaggle Dataset files and save them to the folder path with the file name.
for file in dataset_files:
    file_path = os.path.join(folder_path, file.name)
    
    if not os.path.exists(file_path):
        api.dataset_download_file(
            'lovishbansal123/sales-of-a-supermarket',
            file_name=file.name,
            path=folder_path
        )
        print(f"Downloaded: {file.name}")
    else:
        print(f"Already exists, skipping: {file.name}")

Dataset URL: https://www.kaggle.com/datasets/lovishbansal123/sales-of-a-supermarket
Downloaded: supermarket_sales.csv


In [72]:
## table to log loaded files
conn.execute("""
    CREATE TABLE IF NOT EXISTS loaded_files (
        file_name TEXT PRIMARY KEY,
        load_timestamp TIMESTAMP
    )
""")
conn.commit()

loaded_files = pd.read_sql("SELECT file_name FROM loaded_files", conn)
loaded_files_set = set(loaded_files["file_name"])

In [73]:
from datetime import datetime


all_dataframes = []
raw_df = pd.DataFrame()

## checks for files, skips already loaded files, logs loaded files.
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        

        if file in loaded_files_set:
            print(f"Skipping already loaded file: {file}")
            continue
        
        print(f"Loading new file: {file}")
        
        raw_df = pd.read_csv(file_path)
        all_dataframes.append(raw_df)
        
        # Mark as loaded
        conn.execute("INSERT OR IGNORE INTO loaded_files (file_name, load_timestamp) VALUES (?, ?)", (file,datetime.now().isoformat()))

## combines loaded dataframes and writes all new data to branze. 
if not all_dataframes:
    print("No new files found.")
else:
    raw_df = pd.concat(all_dataframes, ignore_index=True)
    raw_df.to_parquet(os.path.join(bronze_path, "bronze_sales.parquet"), engine='fastparquet', index=False)


Skipping already loaded file: supermarket_sales.csv
No new files found.


## Silver, Data Quality & Quarantine

In [74]:
## changes all naming to lower and snake case
def column_renaming(df):
    new_columns = []
    for col in df.columns:
        if isinstance(col, str):
            new_col = col.strip().lower().replace(' ', '_')
        else:
            new_col = col
        new_columns.append(new_col)
    df.columns = new_columns
    return df

In [75]:
# data quality columns
null_col = ['Invoice ID', 'Customer type', 'Gender', 'Total', 'Date']
dupe_col = ["Invoice ID"]
num_col = ["Total", "Unit price", "Quantity"]

# collecting quarantined records
quarantine = []


if not raw_df.empty:
    for col in num_col:
         null_check = raw_df[col].isnull()
         df_null = raw_df[null_check].copy()
         df_null["quarantine_reason"] = f"Null value in column: {col}"
         quarantine.append(df_null)


    dupes = raw_df.duplicated(subset=dupe_col, keep='first')
    df_dupes = raw_df[dupes].copy()
    df_dupes["quarantine_reason"] = f"Duplicate record based on Invoice ID"
    quarantine.append(df_dupes)


    for col in num_col:
         neg_check = raw_df[col] < 0
         df_neg = raw_df[neg_check].copy()
         df_neg["quarantine_reason"] = f"Negative value found in {col}]"
         quarantine.append(df_neg)

    ## 
    non_dupe_quarantine = [q for q in quarantine if not q.empty and 'Duplicate' not in q['quarantine_reason'].values[0]] if quarantine else []
    bad_ids = pd.concat(non_dupe_quarantine)['Invoice ID'].unique() if non_dupe_quarantine else []

    ## pulling clean records into df_clean, cleaning, writing to silver
    df_clean = raw_df[~raw_df['Invoice ID'].isin(bad_ids)]
    df_clean = column_renaming(df_clean)
    df_clean.to_parquet(os.path.join(silver_path, "silver_sales.parquet"), engine='fastparquet')

    ## combing quarantine records
    df_quarantine = pd.concat(quarantine).drop_duplicates(subset=['Invoice ID'])
    df_quarantine = column_renaming(df_quarantine)

    print(f"Passed to silver: {len(df_clean)}")
    print(f"Quarantined - duplicates: {len(df_dupes)}")
    print(f"Total quarantined: {len(df_quarantine)}")


## writing to Quarantine Parquet
    if not df_quarantine.empty:
        df_quarantine.to_parquet(os.path.join(quarantine_path, 'quarantine_sales.parquet'), engine='fastparquet')
        print("Quarantine file saved!")
else:
     df_clean = column_renaming(raw_df)
     print("No new records to add.")

No new records to add.


## Gold Dimensions


 Create Customer dimension

In [76]:
table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_customer';"
    ).fetchone()
if table_exists:
    print("Table exists")
else:
    conn.execute(""""
    CREATE TABLE IF NOT EXISTS dim_customer (
    cust_id INTEGER PRIMARY KEY,
    customer_type TEXT,
    gender TEXT);"""
    )
    print("Table created")


Table exists


In [77]:
if not df_clean.empty:
    
    new_customer = df_clean[["customer_type", "gender"]].drop_duplicates().reset_index(drop=True)
    existing_customer = pd.read_sql("SELECT * FROM dim_customer", conn)


    if existing_customer.empty:
        new_customer["cust_id"] = new_customer.index + 1
        silver_customer_df = new_customer[["cust_id", "customer_type", "gender"]]
        ## write to dim_customer
        silver_customer_df.to_sql(
            "dim_customer", 
            conn, 
            if_exists="append", 
            index=False
            )
        ## write to Gold 
        silver_customer_df.to_parquet(os.path.join(gold_path, 'dim_customer.parquet'), engine='fastparquet')

    else:
        comparison_df = new_customer.merge(existing_customer[["customer_type", "gender"]],
                        on= ["customer_type", "gender"],
                        how="left", indicator=True)
        new_customer_df = comparison_df[comparison_df["_merge"]=="left_only"].drop(columns=["_merge"])
        ## incremental load 
        if not new_customer_df.empty:
            max_id = existing_customer["cust_id"].max()
            new_customer_df["cust_id"] = range(max_id +1, max_id + 1+ len(new_customer_df))
            new_customer_df.to_sql(
                "dim_customer", 
                conn, 
                if_exists="append", 
                index=False
                )
            new_customer_df.to_parquet(os.path.join(gold_path, 'dim_customer.parquet'), engine='fastparquet')
            silver_customer_df = pd.concat([existing_customer, new_customer_df], ignore_index=True)
        else:
            silver_customer_df = existing_customer
            print("No new customers.")
else:
    print("No new records to process")


No new records to process



 Create Store dimension

In [78]:
table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_store';"
    ).fetchone()
if table_exists:
    print("Table exists")
else:
    conn.execute(""""
    CREATE TABLE IF NOT EXISTS dim_store (
    store_id INTEGER PRIMARY KEY,
    branch TEXT,
    city TEXT
    );"""
    )
    print("Table created")

Table exists


In [79]:
# # checking for data in df_clean
if not df_clean.empty:

    new_store = df_clean[["branch", "city"]].drop_duplicates().reset_index(drop=True)
    existing_store = pd.read_sql("SELECT * FROM dim_store", conn)
    ## initial load
    if existing_store.empty:
        new_store["store_id"] = new_store.index + 1
        ## write to dim_store
        new_store.to_sql(
            "dim_store",
            conn,
            if_exists="append",
            index=False
        )
        ## write to Gold 
        new_store.to_parquet(os.path.join(gold_path, 'dim_store.parquet'), engine='fastparquet')
        print(f"Initial load - {len(new_store)} records.")

    else:
        new_store_df = new_store.merge(existing_store[["branch", "city"]],
                                        on=["branch", "city"],
                                        how="left", indicator=True
                                        )
        new_store_df = new_store_df[new_store_df["_merge"]== "left_only"].drop(columns=["_merge"])

        ## incremental load 
        if not new_store_df.empty:
            max_id = existing_store["store_id"].max()
            new_store_df["store_id"] = range(max_id +1, max_id + 1 + len(new_store_df))

            new_store_df = new_store_df[["store_id", "branch", "city"]]
        
            new_store_df.to_sql(
                "dim_store",
                conn,
                if_exists="append",
                index=False
            )
            new_store_df.to_parquet(os.path.join(gold_path, 'dim_store.parquet'), engine='fastparquet')

            print(f"{len(new_store_df)} new records added.")
        else:
            print("No new stores.")

else:
    print("No new records to process")


No new records to process


Create Product Dimension

In [ ]:
table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_product';"
    ).fetchone()
if table_exists:
    print("Table exists")
else:
    conn.execute(""""
    CREATE TABLE IF NOT EXISTS dim_product (
    cat_id INTEGER PRIMARY KEY,
    product_line TEXT
);"""
    )
    print("Table created")

Table exists


In [81]:
## checking for data in df_clean
if not df_clean.empty:
    new_product = df_clean[["product_line"]].drop_duplicates().reset_index(drop=True)
    existing_product = pd.read_sql("SELECT * FROM dim_product", conn)

    if existing_product.empty:
        new_product["cat_id"] = new_product.index + 1
        new_product = new_product[["cat_id", "product_line"]]
        ## write to dim_product
        new_product.to_sql(
            "dim_product",
            conn,
            if_exists="append",
            index=False
        )
        ## write to Gold 
        new_product.to_parquet(os.path.join(gold_path, 'dim_product.parquet'), engine='fastparquet')
        print(f"Initial load - {len(new_product)} records.")
    else:

        comparison_df = new_product.merge(existing_product[["product_line"]],
                                        on=["product_line"],
                                        how="left", indicator=True)
        new_product_df = comparison_df[comparison_df["_merge"]== "left_only"].drop(columns=["_merge"])
        ## incremental load
        if not new_product_df.empty:
            max_id = existing_product["cat_id"].max()
            new_product_df["cat_id"] = range(max_id + 1, max_id + 1 + len(new_product_df))
            new_product_df = new_product_df[["cat_id", "product_line"]]
            new_product_df.to_sql(
                "dim_product", 
                conn, 
                if_exists="append", 
                index=False
                )
            new_product_df.to_parquet(os.path.join(gold_path, 'dim_product.parquet'), engine='fastparquet')
            print(f"Incremental load - {len(new_product_df)} new records added.")
        
        else:
            print("No new products.")


else:
    print("No new records to process")



No new records to process


Create Date Dimension

In [82]:
table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_date';"
    ).fetchone()
if table_exists:
    print("Table exists")
else:
    conn.execute(""""
    CREATE TABLE IF NOT EXISTS dim_date (
        date_id INTEGER PRIMARY KEY,
        date TEXT,
        day INTEGER,
        month INTEGER,
        year INTEGER,
        quarter INTEGER,
        day_of_week TEXT,
        month_name TEXT,
        week_of_year INTEGER
    );"""
    )
    print("Table created")

Table exists


In [83]:
if not df_clean.empty:
    new_date = df_clean[["date"]].drop_duplicates().reset_index(drop=True)
    new_date["date"] = pd.to_datetime(new_date["date"], errors="coerce")
    existing_date = pd.read_sql("SELECT * FROM dim_date", conn)
    existing_date["date"] = pd.to_datetime(existing_date["date"], errors="coerce")

    if existing_date.empty:
        ## initial load
        dim_date_df = new_date.copy()
        dim_date_df['date_id'] = dim_date_df['date'].dt.strftime('%Y%m%d').astype(int)
        dim_date_df['day'] = dim_date_df['date'].dt.day
        dim_date_df['month'] = dim_date_df['date'].dt.month
        dim_date_df['year'] = dim_date_df['date'].dt.year
        dim_date_df['quarter'] = dim_date_df['date'].dt.quarter
        dim_date_df['day_of_week'] = dim_date_df['date'].dt.day_name()
        dim_date_df['month_name'] = dim_date_df['date'].dt.month_name()
        dim_date_df['week_of_year'] = dim_date_df['date'].dt.isocalendar().week
        dim_date_df['date'] = dim_date_df['date'].dt.date
        dim_date_df.to_sql("dim_date", conn, if_exists="append", index=False)
        dim_date_df.to_parquet(os.path.join(gold_path, 'dim_date.parquet'), engine='fastparquet')
        print(f"Initial load - {len(dim_date_df)} records.")

    else:
        ## incremental load
        new_date_df = new_date[~new_date['date'].isin(existing_date['date'])]

        if not new_date_df.empty:
            dim_date_df = new_date_df.copy()
            dim_date_df['date_id'] = dim_date_df['date'].dt.strftime('%Y%m%d').astype(int)
            dim_date_df['day'] = dim_date_df['date'].dt.day
            dim_date_df['month'] = dim_date_df['date'].dt.month
            dim_date_df['year'] = dim_date_df['date'].dt.year
            dim_date_df['quarter'] = dim_date_df['date'].dt.quarter
            dim_date_df['day_of_week'] = dim_date_df['date'].dt.day_name()
            dim_date_df['month_name'] = dim_date_df['date'].dt.month_name()
            dim_date_df['week_of_year'] = dim_date_df['date'].dt.isocalendar().week
            dim_date_df['date'] = dim_date_df['date'].dt.date
            dim_date_df.to_sql("dim_date", conn, if_exists="append", index=False)
            dim_date_df.to_parquet(os.path.join(gold_path, 'dim_date.parquet'), engine='fastparquet')
            print(f"Incremental load - {len(dim_date_df)} new records added.")
        else:
            print("No new dates to add.")

else:
    print("No new records to process")

No new records to process


## Gold Fact Table

Create Sales Fact Table

In [84]:
### NEW
##### NEW
table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='fact_sales';"
    ).fetchone()
if table_exists:
    print("Table exists")
else:
    conn.execute(""""
    CREATE TABLE IF NOT EXISTS fact_sales (
        invoice_id TEXT PRIMARY KEY,
        store_id INTEGER,
        cust_id INTEGER,
        date_id INTEGER,
        cat_id INTEGER,
        unit_price REAL,
        quantity INTEGER,
        tax_5 REAL,
        total REAL,
        time TEXT,
        payment TEXT,
        cogs REAL,
        gross_margin_percentage REAL,
        gross_income REAL,
        rating REAL,
        FOREIGN KEY (store_id) REFERENCES dim_store(store_id),
        FOREIGN KEY (cust_id) REFERENCES dim_customer(cust_id),
        FOREIGN KEY (date_id) REFERENCES dim_date(date_id),
        FOREIGN KEY (cat_id) REFERENCES dim_product(cat_id)
)   ;"""
    )
    print("Table created")

Table exists


In [85]:
if not df_clean.empty:

    new_sales = df_clean[
        ["invoice_id", "customer_type", "gender", "unit_price", "product_line", "branch", "city", "quantity", "tax_5%", "total", "date", 
         "time", "payment", "cogs", "gross_margin_percentage", "gross_income", "rating"]
    ].drop_duplicates().reset_index(drop=True)

    existing_fact = pd.read_sql("SELECT * FROM fact_sales", conn)

    if existing_fact.empty:
        ## initial load
        dim_store_df = pd.read_sql("SELECT * FROM dim_store", conn)
        dim_customer_df = pd.read_sql("SELECT * FROM dim_customer", conn)
        dim_date_df = pd.read_sql("SELECT * FROM dim_date", conn)
        dim_product_df = pd.read_sql("SELECT * FROM dim_product", conn)

        new_sales['date'] = pd.to_datetime(new_sales['date'], errors='coerce')
        dim_date_df["date"] = pd.to_datetime(dim_date_df["date"], errors='coerce')

        new_sales = new_sales.merge(dim_store_df[['store_id', 'branch', 'city']], on=['branch', 'city'], how='left')
        new_sales = new_sales.merge(dim_customer_df[['cust_id', 'customer_type', 'gender']], on=['customer_type', 'gender'], how='left')
        new_sales = new_sales.merge(dim_date_df[['date_id', 'date']], on='date', how='left')
        new_sales = new_sales.merge(dim_product_df[['cat_id', 'product_line']], on='product_line', how='left')

        fact_columns = ['invoice_id', 'store_id', 'cust_id', 'date_id', 'cat_id',
                        'unit_price', 'quantity', 'tax_5%', 'total', 'time', 'payment',
                        'cogs', 'gross_margin_percentage', 'gross_income', 'rating']

        new_sales = new_sales[fact_columns]
        new_sales.to_sql("fact_sales", conn, if_exists="append", index=False)
        new_sales.to_parquet(os.path.join(gold_path, 'fact_sales.parquet'), engine='fastparquet')
        print(f"Initial load - {len(new_sales)} records.")

    else:
        ## incremental load
        new_sales_df = new_sales[~new_sales["invoice_id"].isin(existing_fact["invoice_id"])]

        if not new_sales_df.empty:
            dim_store_df = pd.read_sql("SELECT * FROM dim_store", conn)
            dim_customer_df = pd.read_sql("SELECT * FROM dim_customer", conn)
            dim_date_df = pd.read_sql("SELECT * FROM dim_date", conn)
            dim_product_df = pd.read_sql("SELECT * FROM dim_product", conn)

            new_sales_df['date'] = pd.to_datetime(new_sales_df['date'], errors='coerce')
            dim_date_df["date"] = pd.to_datetime(dim_date_df["date"], errors='coerce')
            
            # joins
            new_sales_df = new_sales_df.merge(dim_store_df[['store_id', 'branch', 'city']], on=['branch', 'city'], how='left')
            new_sales_df = new_sales_df.merge(dim_customer_df[['cust_id', 'customer_type', 'gender']], on=['customer_type', 'gender'], how='left')
            new_sales_df = new_sales_df.merge(dim_date_df[['date_id', 'date']], on='date', how='left')
            new_sales_df = new_sales_df.merge(dim_product_df[['cat_id', 'product_line']], on='product_line', how='left')

            fact_columns = ['invoice_id', 'store_id', 'cust_id', 'date_id', 'cat_id',
                            'unit_price', 'quantity', 'tax_5%', 'total', 'time', 'payment',
                            'cogs', 'gross_margin_percentage', 'gross_income', 'rating']

            new_sales_df = new_sales_df[fact_columns]
            new_sales_df.to_sql("fact_sales", conn, if_exists="append", index=False)
            new_sales_df.to_parquet(os.path.join(gold_path, 'fact_sales.parquet'), engine='fastparquet')
            print(f"Incremental load - {len(new_sales_df)} new records added.")
        else:
            print("No new invoices.")

else:
    print("No new records to process")

No new records to process


In [86]:
conn.commit()
conn.close()

## Gold KPIs

In [87]:
def run_query(filename):
    with open(filename, "r") as f:
        query = f.read()
    with sqlite3.connect("supermarket_sales.db") as conn:
        return pd.read_sql(query, conn)

In [88]:
## Revenue and Income by Store
revenue_by_store = run_query("SQL Queries\Reveune and Income by Store.sql")
print("KPI 1 - Revenue and Income by Store")
print(revenue_by_store)
revenue_by_store.to_parquet(os.path.join(gold_path, "gold_revenue_by_store.parquet"), engine='fastparquet', index=False)
with sqlite3.connect("supermarket_sales.db") as conn:
    revenue_by_store.to_sql("gold_revenue_by_store", conn, if_exists="replace", index=False)

KPI 1 - Revenue and Income by Store
         city branch            product_line  total_revenue  \
0    Mandalay      B       Sports and travel       19988.20   
1    Mandalay      B       Health and beauty       19980.66   
2    Mandalay      B      Home and lifestyle       17549.16   
3    Mandalay      B  Electronic accessories       17051.44   
4    Mandalay      B     Fashion accessories       16413.32   
5    Mandalay      B      Food and beverages       15214.89   
6   Naypyitaw      C      Food and beverages       23766.85   
7   Naypyitaw      C     Fashion accessories       21560.07   
8   Naypyitaw      C  Electronic accessories       18968.97   
9   Naypyitaw      C       Health and beauty       16615.33   
10  Naypyitaw      C       Sports and travel       15761.93   
11  Naypyitaw      C      Home and lifestyle       13895.55   
12     Yangon      A      Home and lifestyle       22417.20   
13     Yangon      A       Sports and travel       19372.70   
14     Yangon      

<>:2: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
<>:2: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
C:\Users\theol\AppData\Local\Temp\ipykernel_10856\962939004.py:2: SyntaxWarning: "\R" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\R"? A raw string is also an option.
  revenue_by_store = run_query("SQL Queries\Reveune and Income by Store.sql")


In [89]:
## Average Transaction by Store, Product, Month
avg_transaction_by_store = run_query("SQL Queries\Average Transaction by Store, Product, Month.sql")
print("\n KPI 2 - Average Transaction by Store, Product, Month")
print(avg_transaction_by_store)
avg_transaction_by_store.to_parquet(os.path.join(gold_path, "gold_avg_transaction_by_store.parquet"), engine='fastparquet', index=False)
with sqlite3.connect("supermarket_sales.db") as conn:
    avg_transaction_by_store.to_sql("gold_avg_transaction_by_store", conn, if_exists="replace", index=False)


 KPI 2 - Average Transaction by Store, Product, Month
    month       city            product_line  avg_trans
0       1   Mandalay       Health and beauty     399.99
1       1   Mandalay  Electronic accessories     372.21
2       1   Mandalay      Food and beverages     347.86
3       2   Mandalay       Health and beauty     366.03
4       2   Mandalay  Electronic accessories     351.91
5       2   Mandalay      Home and lifestyle     332.85
6       3   Mandalay       Sports and travel     384.52
7       3   Mandalay      Home and lifestyle     377.40
8       3   Mandalay       Health and beauty     367.83
9       1  Naypyitaw  Electronic accessories     382.02
10      1  Naypyitaw      Food and beverages     377.96
11      1  Naypyitaw       Sports and travel     364.74
12      2  Naypyitaw     Fashion accessories     384.96
13      2  Naypyitaw       Sports and travel     353.74
14      2  Naypyitaw       Health and beauty     323.91
15      3  Naypyitaw      Food and beverages     

<>:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\theol\AppData\Local\Temp\ipykernel_10856\3559108941.py:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
  avg_transaction_by_store = run_query("SQL Queries\Average Transaction by Store, Product, Month.sql")


In [90]:
## Monthly Revenue by Branch
monthly_revenue_by_branch = run_query("SQL Queries\Monthly Revenue by Branch.sql")
print("\nKPI 3 - Monthly Revenue by Branch")
print(monthly_revenue_by_branch)
monthly_revenue_by_branch.to_parquet(os.path.join(gold_path, "gold_monthly_revenue_by_branch.parquet"), engine='fastparquet', index=False)
with sqlite3.connect("supermarket_sales.db") as conn:
    monthly_revenue_by_branch.to_sql("gold_monthly_revenue_by_branch", conn, if_exists="replace", index=False)


KPI 3 - Monthly Revenue by Branch
        city branch month_name  total_revenue  prev_month_revenue  \
0   Mandalay      B    January       37176.06                 NaN   
1   Mandalay      B   February       34424.27            37176.06   
2   Mandalay      B      March       34597.34            34424.27   
3  Naypyitaw      C    January       40434.68                 NaN   
4  Naypyitaw      C   February       32934.98            40434.68   
5  Naypyitaw      C      March       37199.04            32934.98   
6     Yangon      A    January       38681.13                 NaN   
7     Yangon      A   February       29860.12            38681.13   
8     Yangon      A      March       37659.12            29860.12   

   revenue_change trend  
0             NaN   N/A  
1        -2751.79  Down  
2          173.07    Up  
3             NaN   N/A  
4        -7499.70  Down  
5         4264.06    Up  
6             NaN   N/A  
7        -8821.01  Down  
8         7799.00    Up  


<>:2: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:2: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\theol\AppData\Local\Temp\ipykernel_10856\1139165185.py:2: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  monthly_revenue_by_branch = run_query("SQL Queries\Monthly Revenue by Branch.sql")


In [91]:
## Average Rating by Category and Branch
avg_rating_by_category = run_query("SQL Queries\Average Rating by Category and Branch.sql")
print("\nKPI 4 - Average Rating by Category and Branch")
print(avg_rating_by_category)
avg_rating_by_category.to_parquet(os.path.join(gold_path, "gold_avg_rating_by_category.parquet"), engine='fastparquet', index=False)
with sqlite3.connect("supermarket_sales.db") as conn:
    avg_rating_by_category.to_sql("gold_avg_rating_by_category", conn, if_exists="replace", index=False)


KPI 4 - Average Rating by Category and Branch
         city branch            product_line  avg_rating  transaction_count  \
0    Mandalay      B  Electronic accessories        7.12                 55   
1    Mandalay      B       Health and beauty        7.10                 53   
2    Mandalay      B      Food and beverages        6.99                 50   
3    Mandalay      B     Fashion accessories        6.72                 62   
4    Mandalay      B      Home and lifestyle        6.52                 50   
5    Mandalay      B       Sports and travel        6.51                 62   
6   Naypyitaw      C     Fashion accessories        7.44                 65   
7   Naypyitaw      C      Food and beverages        7.08                 66   
8   Naypyitaw      C      Home and lifestyle        7.06                 45   
9   Naypyitaw      C       Sports and travel        7.03                 45   
10  Naypyitaw      C       Health and beauty        7.00                 52   
11  N

<>:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\theol\AppData\Local\Temp\ipykernel_10856\203407810.py:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
  avg_rating_by_category = run_query("SQL Queries\Average Rating by Category and Branch.sql")
